In [0]:

# Display all tables available in the Bronze schema
display(spark.sql("SHOW TABLES IN courier_delivery.bronze"))


In [0]:

# Import PySpark functions required for Silver transformations
from pyspark.sql.functions import (
    col, trim, upper, to_date, when, concat_ws, lit
)

# Import data types for converting columns
from pyspark.sql.types import IntegerType, DecimalType

# Load the initial raw records from Bronze
initial_bronze = spark.table("courier_delivery.bronze.orders_initial_raw")

# Load the incremental raw records from Bronze
incremental_bronze = spark.table("courier_delivery.bronze.orders_incremental_raw")

# Check the record counts in both batches
print("Initial Bronze records:", initial_bronze.count())
print("Incremental Bronze records:", incremental_bronze.count())


In [0]:

# Apply data-quality checks to the initial batch
initial_checked = initial_bronze.withColumn(
    "rejection_reason",
    concat_ws(", ",
        when(col("order_id").isNull(), lit("Missing order_id")),
        when(col("customer_id").isNull(), lit("Missing customer_id")),
        when(col("product_id").isNull(), lit("Missing product_id")),
        when(col("quantity").isNull() | (col("quantity") <= 0), lit("Invalid quantity")),
        when(col("unit_price").isNull() | (col("unit_price") < 0), lit("Invalid unit_price")),
        when(col("sales_amount").isNull() | (col("sales_amount") < 0), lit("Invalid sales_amount"))
    )
)

# Keep only initial records that passed validation
initial_valid = initial_checked.filter(col("rejection_reason") == "")

# Apply the same validation checks to the incremental batch
incremental_checked = incremental_bronze.withColumn(
    "rejection_reason",
    concat_ws(", ",
        when(col("order_id").isNull(), lit("Missing order_id")),
        when(col("customer_id").isNull(), lit("Missing customer_id")),
        when(col("product_id").isNull(), lit("Missing product_id")),
        when(col("quantity").isNull() | (col("quantity") <= 0), lit("Invalid quantity")),
        when(col("unit_price").isNull() | (col("unit_price") < 0), lit("Invalid unit_price")),
        when(col("sales_amount").isNull() | (col("sales_amount") < 0), lit("Invalid sales_amount"))
    )
)

# Keep only incremental records that passed validation
incremental_valid = incremental_checked.filter(col("rejection_reason") == "")

# Display the valid record counts
print("Valid initial records:", initial_valid.count())
print("Valid incremental records:", incremental_valid.count())


In [0]:

# Remove the rejection_reason column because these records are valid
silver_initial = initial_valid.drop("rejection_reason")

# Standardise city names by removing spaces and converting to uppercase
silver_initial = silver_initial.withColumn("city", upper(trim(col("city"))))

# Standardise payment methods
silver_initial = silver_initial.withColumn("payment_method", upper(trim(col("payment_method"))))

# Standardise order statuses
silver_initial = silver_initial.withColumn("order_status", upper(trim(col("order_status"))))

# Convert order_date into a proper date type
silver_initial = silver_initial.withColumn("order_date", to_date(col("order_date")))

# Convert quantity into an integer
silver_initial = silver_initial.withColumn("quantity", col("quantity").cast(IntegerType()))

# Convert unit_price into a decimal with two decimal places
silver_initial = silver_initial.withColumn("unit_price", col("unit_price").cast(DecimalType(12, 2)))

# Convert discount_pct into a decimal with two decimal places
silver_initial = silver_initial.withColumn("discount_pct", col("discount_pct").cast(DecimalType(5, 2)))

# Convert sales_amount into a decimal with two decimal places
silver_initial = silver_initial.withColumn("sales_amount", col("sales_amount").cast(DecimalType(12, 2)))

# Remove duplicate order IDs
silver_initial = silver_initial.dropDuplicates(["order_id"])

# Display the cleaned initial records
display(silver_initial)

# Print the final initial record count
print("Clean initial records:", silver_initial.count())


In [0]:

# Save the cleaned initial records as a Delta table
silver_initial.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("courier_delivery.silver.orders")

# Read the saved Silver table
silver_orders = spark.table("courier_delivery.silver.orders")

# Display the saved records
display(silver_orders)

# Verify the number of records stored
print("Records in Silver table:", silver_orders.count())


In [0]:

# Select only incremental records that passed Bronze validation
silver_incremental = incremental_valid.drop("rejection_reason")

# Standardise city names by trimming spaces and converting to uppercase
silver_incremental = silver_incremental.withColumn(
    "city", upper(trim(col("city")))
)

# Standardise payment methods
silver_incremental = silver_incremental.withColumn(
    "payment_method", upper(trim(col("payment_method")))
)

# Standardise order statuses
silver_incremental = silver_incremental.withColumn(
    "order_status", upper(trim(col("order_status")))
)

# Convert order_date into a date data type
silver_incremental = silver_incremental.withColumn(
    "order_date", to_date(col("order_date"))
)

# Convert quantity into an integer
silver_incremental = silver_incremental.withColumn(
    "quantity", col("quantity").cast(IntegerType())
)

# Convert monetary and percentage columns into decimal data types
silver_incremental = silver_incremental.withColumn(
    "unit_price", col("unit_price").cast(DecimalType(12, 2))
)
silver_incremental = silver_incremental.withColumn(
    "discount_pct", col("discount_pct").cast(DecimalType(5, 2))
)
silver_incremental = silver_incremental.withColumn(
    "sales_amount", col("sales_amount").cast(DecimalType(12, 2))
)

# Remove duplicate order IDs within the incremental batch
silver_incremental = silver_incremental.dropDuplicates(["order_id"])

# Display the cleaned incremental records
display(silver_incremental)

# Verify the incremental record count
print("Clean incremental records:", silver_incremental.count())


In [0]:

# Import Delta Lake support for performing MERGE operations
from delta.tables import DeltaTable

# Identify the existing Silver Delta table
silver_table = DeltaTable.forName(spark, "courier_delivery.silver.orders")

# Create a temporary view from the cleaned incremental DataFrame
silver_incremental.createOrReplaceTempView("incremental_orders")

# Merge incremental records into the Silver table using order_id as the business key
spark.sql("""
    MERGE INTO courier_delivery.silver.orders AS target
    USING incremental_orders AS source
    ON target.order_id = source.order_id

    WHEN MATCHED AND source.updated_at > target.updated_at THEN
        UPDATE SET *

    WHEN NOT MATCHED THEN
        INSERT *
""")

# Read the updated Silver table
silver_orders = spark.table("courier_delivery.silver.orders")

# Display the final Silver records
display(silver_orders)

# Print the final number of records after the merge
print("Final Silver record count:", silver_orders.count())


In [0]:

# Read the final Silver table
silver_orders = spark.table("courier_delivery.silver.orders")

# Group records by order_id and identify duplicates
duplicate_orders = (
    silver_orders
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

# Display duplicate records, if any
display(duplicate_orders)

# Print the total number of duplicate order IDs
print("Duplicate order IDs:", duplicate_orders.count())

# Print the final number of Silver records
print("Final Silver records:", silver_orders.count())


In [0]:

# Read the final Silver table
silver_orders = spark.table("courier_delivery.silver.orders")

# Display the table structure
silver_orders.printSchema()

# Check the order IDs and update timestamps
display(
    silver_orders.select(
        "order_id",
        "order_status",
        "sales_amount",
        "updated_at"
    ).orderBy("order_id")
)

# Count records with a missing updated_at value
print(
    "Missing updated_at values:",
    silver_orders.filter(col("updated_at").isNull()).count()
)


In [0]:
# Count records where updated_at is missing
print(
    "Missing updated_at values:",
    silver_orders.filter(col("updated_at").isNull()).count()
)

# Display the Silver table column names and data types
silver_orders.printSchema()

In [0]:
# Count records where updated_at is missing
print(
    "Missing updated_at values:",
    silver_orders.filter(col("updated_at").isNull()).count()
)

In [0]:
%sql
SELECT COUNT(*)
FROM courier_delivery.dbt_dev.stg_orders;

In [0]:
%sql
-- Check that the snapshot table exists
SHOW TABLES IN courier_delivery.snapshot;

-- Check the number of records captured
SELECT COUNT(*)
FROM courier_delivery.snapshot.orders_snapshot;

-- View the SCD Type 2 history columns
SELECT
    order_id,
    order_status,
    updated_at,
    dbt_valid_from,
    dbt_valid_to
FROM courier_delivery.snapshot.orders_snapshot
LIMIT 10;

In [0]:
# Import functions required for validation and transformation
from pyspark.sql.functions import col, lit, trim

# Load the initial and incremental Bronze tables
initial_df = spark.table("courier_delivery.bronze.orders_initial_raw")
incremental_df = spark.table("courier_delivery.bronze.orders_incremental_raw")

# Combine both Bronze batches
all_bronze = initial_df.unionByName(incremental_df, allowMissingColumns=True)

# Identify records where order_status is NULL or blank
missing_status = all_bronze.filter(
    col("order_status").isNull() | (trim(col("order_status")) == "")
)

# Remove duplicate order IDs from the records being rejected
missing_status = missing_status.dropDuplicates(["order_id"])

# Load the existing Bronze rejects table
existing_rejects = spark.table("courier_delivery.bronze.orders_rejects")

# Exclude records that are already present in the rejects table
existing_ids = existing_rejects.select("order_id").distinct()

new_rejects = missing_status.join(existing_ids, on="order_id", how="left_anti")

# Add the rejection reason
new_rejects = new_rejects.withColumn(
    "rejection_reason",
    lit("Missing or blank order_status")
)

# Match the existing rejects table's column structure
reject_columns = existing_rejects.columns

for column_name in reject_columns:
    if column_name not in new_rejects.columns:
        new_rejects = new_rejects.withColumn(column_name, lit(None))

new_rejects = new_rejects.select(*reject_columns)

# Append the newly rejected records to the Bronze rejects table
new_rejects.write.format("delta").mode("append").saveAsTable(
    "courier_delivery.bronze.orders_rejects"
)

# Display how many records were newly rejected
print("Newly rejected records:", new_rejects.count())

In [0]:
%sql
-- Remove orders with missing or blank status from Silver
DELETE FROM courier_delivery.silver.orders
WHERE order_status IS NULL
   OR TRIM(order_status) = '';

-- Verify that no missing statuses remain
SELECT COUNT(*) AS remaining_missing_status
FROM courier_delivery.silver.orders
WHERE order_status IS NULL
   OR TRIM(order_status) = '';

In [0]:
%sql
-- Check the updated total number of rejected records
SELECT COUNT(*) AS total_rejected_records
FROM courier_delivery.bronze.orders_rejects;

-- Check the rejection reasons
SELECT
    rejection_reason,
    COUNT(*) AS rejected_count
FROM courier_delivery.bronze.orders_rejects
GROUP BY rejection_reason
ORDER BY rejected_count DESC;

In [0]:
%sql
-- Display all tables in the Gold schema
SHOW TABLES IN courier_delivery.gold;

-- Check the number of records in each Gold table
SELECT COUNT(*) AS city_records
FROM courier_delivery.gold.gold_sales_by_city;

SELECT COUNT(*) AS product_records
FROM courier_delivery.gold.gold_sales_by_product;

SELECT COUNT(*) AS status_records
FROM courier_delivery.gold.gold_orders_by_status;

In [0]:
%sql
-- Check the default dbt development schema
SHOW TABLES IN courier_delivery.dbt_dev;

-- Check whether dbt created a separate Gold schema
SHOW SCHEMAS IN courier_delivery;

In [0]:
%sql
-- List tables created by dbt in its generated Gold schema
SHOW TABLES IN courier_delivery.dbt_dev_gold;

In [0]:
%sql
-- Permanently drop dbt_dev_gold and all objects inside it
DROP SCHEMA IF EXISTS courier_delivery.dbt_dev_gold CASCADE;

In [0]:
%sql
-- Count initial Bronze records
SELECT COUNT(*) AS initial_records
FROM courier_delivery.bronze.orders_initial_raw;

-- Count incremental Bronze records
SELECT COUNT(*) AS incremental_records
FROM courier_delivery.bronze.orders_incremental_raw;

-- Count all rejected records
SELECT COUNT(*) AS rejected_records
FROM courier_delivery.bronze.orders_rejects;

In [0]:
%sql
-- Count final valid records in Silver
SELECT COUNT(*) AS silver_records
FROM courier_delivery.silver.orders;

In [0]:
%sql
-- Check staging model
SELECT COUNT(*) AS staging_records
FROM courier_delivery.dbt_dev.stg_orders;

-- Check Gold models
SELECT * FROM courier_delivery.dbt_dev.gold_sales_by_city;
SELECT * FROM courier_delivery.dbt_dev.gold_sales_by_product;
SELECT * FROM courier_delivery.dbt_dev.gold_orders_by_status;

In [0]:
%sql
-- Check whether the Gold models now exist in dbt_dev
SHOW TABLES IN courier_delivery.dbt_dev;

In [0]:
%sql
-- Check sales by city
SELECT *
FROM courier_delivery.dbt_dev.gold_sales_by_city
ORDER BY total_sales DESC;

-- Check sales by product
SELECT *
FROM courier_delivery.dbt_dev.gold_sales_by_product
ORDER BY total_sales DESC;

-- Check orders by status
SELECT *
FROM courier_delivery.dbt_dev.gold_orders_by_status
ORDER BY total_orders DESC;
